# Drought News Detection
Step-by-step pipeline to collect, process, and classify drought-related news articles.

## Step 1: Install Dependencies

In [ ]:
!pip install requests pandas numpy scikit-learn nltk matplotlib seaborn feedparser tqdm

## Step 2: Imports & Setup

In [ ]:
import requests
import pandas as pd
import numpy as np
import re
import os
import pickle
from tqdm import tqdm
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')
print('Setup complete.')

## Step 3: Define Drought Keywords

In [ ]:
DROUGHT_KEYWORDS = [
    'drought', 'droughts', 'water shortage', 'water scarcity', 'water crisis',
    'dry spell', 'arid', 'desertification', 'water stress', 'water deficit',
    'rainfall deficit', 'precipitation shortage', 'groundwater depletion',
    'reservoir levels', 'water restrictions', 'soil moisture deficit',
    'hydrological drought', 'meteorological drought', 'agricultural drought',
    'parched', 'withered crops'
]
print(f'Keywords: {len(DROUGHT_KEYWORDS)}')

## Step 4: Load Data

Uses built-in sample data. Replace with real news data (NewsAPI / RSS) when ready.

In [ ]:
SAMPLE_DATA = [
    {'title': 'Severe drought threatens crops across southern Australia', 'description': 'Farmers face ruin as water reserves hit record lows amid prolonged dry spell.', 'label': 1},
    {'title': 'California water restrictions tighten as reservoir levels drop', 'description': 'State officials impose mandatory cuts as drought enters third year.', 'label': 1},
    {'title': 'Horn of Africa drought leaves millions facing food crisis', 'description': 'Aid agencies warn of humanitarian disaster as rainfall deficit continues.', 'label': 1},
    {'title': 'Kenya declares national drought emergency', 'description': 'Government seeks international aid as pastures dry up and livestock die.', 'label': 1},
    {'title': 'Water scarcity forces farmers to abandon fields in Spain', 'description': 'Groundwater depletion accelerates across Iberian peninsula.', 'label': 1},
    {'title': 'Amazon basin faces worst drought in recorded history', 'description': 'River levels at century lows, threatening biodiversity and transport.', 'label': 1},
    {'title': 'Indian farmers protest as monsoon fails for second year', 'description': 'Soil moisture deficit leaves millions of acres unplanted across region.', 'label': 1},
    {'title': 'Western US faces multi-year megadrought conditions', 'description': 'Scientists link prolonged water shortage to accelerating climate change.', 'label': 1},
    {'title': 'Ethiopia facing worst water shortage in decades', 'description': 'Communities walk miles for water as wells run dry across the region.', 'label': 1},
    {'title': 'Murray-Darling basin drought devastates Australian agriculture', 'description': 'Hydrological drought declared as river flows hit historic lows.', 'label': 1},
    {'title': 'Zimbabwe declares state of disaster over drought', 'description': 'Crop failures and water shortages leave millions in need of food aid.', 'label': 1},
    {'title': 'Southern Europe braces for another summer of extreme drought', 'description': 'Meteorologists warn rainfall deficit will worsen through dry season.', 'label': 1},
    {'title': 'New smartphone model released with improved camera system', 'description': 'Tech giant unveils latest flagship with triple-lens array and AI features.', 'label': 0},
    {'title': 'Stock markets rally as earnings season beats expectations', 'description': 'Major indices climb on strong corporate results across tech and finance.', 'label': 0},
    {'title': 'Local football team wins championship after dramatic final', 'description': 'Fans celebrate as team clinches title in penalty shootout.', 'label': 0},
    {'title': 'Film festival announces lineup for upcoming season', 'description': 'Directors from 40 countries compete for prestigious international award.', 'label': 0},
    {'title': 'Electric vehicle sales surge in European market', 'description': 'Automakers report record quarterly figures driven by consumer demand.', 'label': 0},
    {'title': 'Scientists discover new species of deep-sea fish', 'description': 'Marine biologists find previously unknown creature at 3000m depth.', 'label': 0},
    {'title': 'Central bank raises interest rates to combat inflation', 'description': 'Policymakers vote to increase benchmark rate by 25 basis points.', 'label': 0},
    {'title': 'Streaming platform announces major original series renewal', 'description': 'Popular drama confirmed for two additional seasons after viewership record.', 'label': 0},
    {'title': 'New museum wing opens showcasing ancient civilisations', 'description': 'Visitors flock to see artefacts spanning three thousand years of history.', 'label': 0},
    {'title': 'City council approves new public transport expansion plan', 'description': 'Rail network to extend to three new districts by 2028.', 'label': 0},
    {'title': 'Tech startup raises record funding round for AI research', 'description': 'Investors back new generative AI company in landmark deal.', 'label': 0},
    {'title': 'Olympic committee announces host city for next games', 'description': 'Decision made after decade-long selection process across candidate cities.', 'label': 0},
]

df = pd.DataFrame(SAMPLE_DATA)
df['text'] = df['title'] + ' ' + df['description']
print(f'Total articles: {len(df)}')
print(df['label'].value_counts().rename({0: 'Other', 1: 'Drought'}))
df.head(3)

## Step 5: Explore the Data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'])
axes[0].set_title('Class Distribution')
axes[0].set_xticklabels(['Other', 'Drought'], rotation=0)
df['text_len'] = df['text'].str.len()
df.groupby('label')['text_len'].plot(kind='hist', bins=15, alpha=0.6, ax=axes[1])
axes[1].set_title('Text Length')
axes[1].legend(['Other', 'Drought'])
plt.tight_layout()
plt.show()

## Step 6: Text Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

tqdm.pandas()
df['clean_text'] = df['text'].progress_apply(preprocess)
print('Before:', df['text'].iloc[0][:80])
print('After: ', df['clean_text'].iloc[0][:80])

## Step 7: Keyword Baseline

In [ ]:
def keyword_classifier(text):
    return int(any(kw in text.lower() for kw in DROUGHT_KEYWORDS))

df['keyword_pred'] = df['text'].apply(keyword_classifier)
print('Keyword Baseline:')
print(classification_report(df['label'], df['keyword_pred'], target_names=['Other', 'Drought']))

## Step 8: TF-IDF + Logistic Regression

In [ ]:
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)

y_pred = lr_model.predict(X_test_tfidf)
print('Logistic Regression:')
print(classification_report(y_test, y_pred, target_names=['Other', 'Drought']))

## Step 9: Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Other', 'Drought'], yticklabels=['Other', 'Drought'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## Step 10: Top Predictive Words

In [ ]:
feature_names = vectorizer.get_feature_names_out()
coef = lr_model.coef_[0]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pd.Series(coef, index=feature_names).nlargest(15).plot(kind='barh', ax=axes[0], color='tomato')
axes[0].set_title('Top Drought Words')
pd.Series(coef, index=feature_names).nsmallest(15).plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top Non-Drought Words')
plt.tight_layout()
plt.show()

## Step 11: Predict New Headlines

In [ ]:
def predict_drought(headlines):
    cleaned = [preprocess(h) for h in headlines]
    tfidf   = vectorizer.transform(cleaned)
    preds   = lr_model.predict(tfidf)
    probs   = lr_model.predict_proba(tfidf)[:, 1]
    return pd.DataFrame({'headline': headlines, 'drought': preds, 'confidence': probs.round(3)})

test_headlines = [
    'Severe drought threatens crops across southern Australia',
    'Stock markets rally on strong earnings reports',
    'Water reservoirs hit record low as dry spell continues',
    'New smartphone model released with improved camera',
    'Farmers face ruin as drought decimates harvest',
]
predict_drought(test_headlines)

## Step 12: Save Model

In [ ]:
os.makedirs('../data', exist_ok=True)
df.to_csv('../data/articles_labeled.csv', index=False)
with open('../data/model.pkl', 'wb') as f:
    pickle.dump({'vectorizer': vectorizer, 'model': lr_model}, f)
print('Saved: data/articles_labeled.csv')
print('Saved: data/model.pkl')

## Next Steps

- Add real news data via NewsAPI or GDELT
- Try BERT / DistilBERT for better accuracy
- Add geographic tagging per article
- Build severity score (mild / moderate / severe)
- Schedule daily fetch + classify pipeline